In [1]:
import os
import torch
import clip
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-B/32"

model, preprocess = clip.load(MODEL_NAME, device=DEVICE)
model.eval()

print("Device:", DEVICE)


Device: cuda


In [2]:
DATA_ROOT = "../segments_16/train"
CLASS_NAME = "Fighting"

class_dir = os.path.join(DATA_ROOT, CLASS_NAME)

video_list = sorted(os.listdir(class_dir))
first_video = video_list[0]
video_dir = os.path.join(class_dir, first_video)

segment_list = sorted(os.listdir(video_dir))
first_segment = segment_list[0]
segment_dir = os.path.join(video_dir, first_segment)

frame_list = sorted(os.listdir(segment_dir))
first_frame = frame_list[0]
image_path = os.path.join(segment_dir, first_frame)

image = Image.open(image_path)

plt.imshow(image)
plt.axis("off")

print("Selected video:", first_video)
print("Selected segment:", first_segment)
print("Frame path:", image_path)


FileNotFoundError: [Errno 2] No such file or directory: '../segments_16/train/Fighting'

In [ ]:
TEXTS = [
    "a person fighting",
    "people walking normally",
    "a peaceful street scene",
    "a violent action",
    "a robbery happening"
]

with torch.no_grad():
    text_tokens = clip.tokenize(TEXTS).to(DEVICE)
    text_embeddings = model.encode_text(text_tokens)
    text_embeddings = F.normalize(text_embeddings, dim=-1)

print("Text embeddings shape:", text_embeddings.shape)


In [ ]:
# Cell 7 – Segment sanity check

print("Using video directory:", video_dir)
print("Using segment directory:", segment_dir)

frame_list = sorted(os.listdir(segment_dir))
print("Total frames in segment:", len(frame_list))
print("First 3 frames:", frame_list[:3])


In [ ]:
def compute_segment_embedding(segment_dir):
    frame_files = sorted(os.listdir(segment_dir))
    frame_embeddings = []

    with torch.no_grad():
        for frame_name in frame_files:
            frame_path = os.path.join(segment_dir, frame_name)

            image = Image.open(frame_path).convert("RGB")
            image_input = preprocess(image).unsqueeze(0).to(DEVICE)

            emb = model.encode_image(image_input)
            frame_embeddings.append(emb)

    frame_embeddings = torch.cat(frame_embeddings, dim=0)

    segment_embedding = frame_embeddings.mean(dim=0, keepdim=True)
    segment_embedding = F.normalize(segment_embedding, dim=-1)

    return segment_embedding
